In [20]:
# ============================================================================
# OPSEE Workflow - Cell 1: Imports and Helper Functions
# ============================================================================
# This cell imports all required libraries and defines helper functions for
# extracting equipment and instrument data from DEXPI XML files using pyDEXPI.
#
# Dependencies:
#   - rocrate: RO-Crate Python library for FAIR packaging
#   - ipywidgets: Interactive UI widgets for Jupyter
#   - pyDEXPI: DEXPI XML parser from process-intelligence-research
#   - Custom modules: rocrate_builder, validators
# ============================================================================

# Standard library imports
import os
from datetime import datetime
from pathlib import Path

# Third-party interactive widgets
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# RO-Crate library imports
from rocrate.rocrate import ROCrate
from rocrate.model.person import Person
from rocrate.model.contextentity import ContextEntity

# Import pyDEXPI directly (no wrapper module)
from pydexpi.loaders import ProteusSerializer

# Custom module imports from src/
import sys
sys.path.insert(0, 'src')
from rocrate_builder import OPSEECrateBuilder

# Import validators module directly to avoid conflicts with pip 'validators' package
import importlib.util
spec = importlib.util.spec_from_file_location("opsee_validators", "src/opsee/validators.py")
opsee_validators = importlib.util.module_from_spec(spec)
spec.loader.exec_module(opsee_validators)
validate_crate = opsee_validators.validate_crate
validate_dexpi = opsee_validators.validate_dexpi


# ============================================================================
# Helper Functions for DEXPI Extraction
# ============================================================================

def extract_equipment(dexpi_model):
    """Extract equipment from pyDEXPI model.
    
    Parses the conceptualModel.taggedPlantItems from a loaded DEXPI model
    and converts to dictionary format for RO-Crate builder consumption.
    
    Args:
        dexpi_model: Loaded DEXPI model from ProteusSerializer.load()
    
    Returns:
        Dictionary mapping equipment IDs to metadata:
        {
            'eq_id': {
                'id': 'unique_identifier',
                'tag_name': 'R-101',
                'type': 'Reactor',
                'name': 'Main Reactor',
                'element': 'Equipment'
            },
            ...
        }
    """
    equipment = {}
    if hasattr(dexpi_model, 'conceptualModel') and dexpi_model.conceptualModel:
        for eq in dexpi_model.conceptualModel.taggedPlantItems:
            # Generate unique ID from DEXPI object
            eq_id = str(eq.id) if hasattr(eq, 'id') else str(id(eq))
            
            # Extract tag name with fallback chain
            tag_name = None
            if hasattr(eq, 'tagName') and eq.tagName:
                tag_name = eq.tagName
            elif hasattr(eq, 'componentName') and eq.componentName:
                tag_name = eq.componentName
            
            # Build equipment metadata dictionary
            equipment[eq_id] = {
                'id': eq_id,
                'tag_name': tag_name or eq_id,  # Fallback to ID if no tag
                'type': eq.__class__.__name__,   # e.g., 'Reactor', 'Tank'
                'name': eq.componentName if hasattr(eq, 'componentName') else '',
                'element': 'Equipment'
            }
    return equipment


def extract_instruments(dexpi_model):
    """Extract instruments from pyDEXPI model.
    
    Parses both actuatingSystems and processSignalGeneratingSystems from
    the DEXPI conceptual model to extract all instrumentation metadata.
    
    Args:
        dexpi_model: Loaded DEXPI model from ProteusSerializer.load()
    
    Returns:
        Dictionary mapping instrument IDs to metadata:
        {
            'inst_id': {
                'id': 'unique_identifier',
                'tag_name': 'GC-001',
                'type': 'ActuatingSystem' | 'ProcessSignalGeneratingSystem',
                'description': 'Gas chromatograph',
                'element': 'ActuatingSystem' | 'MeasuringSystem'
            },
            ...
        }
    """
    instruments = {}
    if hasattr(dexpi_model, 'conceptualModel') and dexpi_model.conceptualModel:
        
        # Extract actuating systems (control instruments, valves)
        for act_sys in dexpi_model.conceptualModel.actuatingSystems:
            inst_id = str(act_sys.id) if hasattr(act_sys, 'id') else str(id(act_sys))
            
            # Extract tag name with multiple fallbacks
            tag_name = None
            if hasattr(act_sys, 'actuatingSystemNumber') and act_sys.actuatingSystemNumber:
                tag_name = act_sys.actuatingSystemNumber
            elif hasattr(act_sys, 'componentName') and act_sys.componentName:
                tag_name = act_sys.componentName
            
            # Extract description if available
            description = ''
            if hasattr(act_sys, 'typicalInformation') and act_sys.typicalInformation:
                description = act_sys.typicalInformation
            
            instruments[inst_id] = {
                'id': inst_id,
                'tag_name': tag_name or inst_id,
                'type': 'ActuatingSystem',
                'description': description,
                'element': 'ActuatingSystem'
            }
        
        # Extract process signal generating systems (measuring instruments)
        if hasattr(dexpi_model.conceptualModel, 'processSignalGeneratingSystems'):
            for psgs in dexpi_model.conceptualModel.processSignalGeneratingSystems:
                inst_id = str(psgs.id) if hasattr(psgs, 'id') else str(id(psgs))
                
                # Extract tag name with multiple fallbacks
                tag_name = None
                if hasattr(psgs, 'processSignalGeneratingSystemNumber') and psgs.processSignalGeneratingSystemNumber:
                    tag_name = psgs.processSignalGeneratingSystemNumber
                elif hasattr(psgs, 'componentName') and psgs.componentName:
                    tag_name = psgs.componentName
                
                # Extract description if available
                description = ''
                if hasattr(psgs, 'typicalInformation') and psgs.typicalInformation:
                    description = psgs.typicalInformation
                
                instruments[inst_id] = {
                    'id': inst_id,
                    'tag_name': tag_name or inst_id,
                    'type': 'ProcessSignalGeneratingSystem',
                    'description': description,
                    'element': 'MeasuringSystem'  # Classified as measuring device
                }
    return instruments


# Success indicator
print("✓ All libraries imported successfully")
print("✓ DEXPI helper functions defined")

✓ All libraries imported successfully
✓ DEXPI helper functions defined


## 0. RO-Crate Output Directory ⚠️ START HERE

**Important:** First, choose where your RO-Crate will be created. All referenced files (DEXPI, data files, assets) will be **COPIED** into this directory to create a fully self-contained, portable package.

This means:
- ✅ Files can be located **anywhere** on your system initially
- ✅ Everything gets **copied** into the RO-Crate directory structure
- ✅ The resulting RO-Crate is **completely portable** - no broken links!
- ✅ You can share/archive the entire directory independently

In [ ]:
# ============================================================================
# RO-Crate Output Directory Configuration
# ============================================================================
# Define where the RO-Crate will be created. All files (DEXPI, data, assets)
# will be COPIED into this directory structure following OPSEE conventions:
#
#   my_rocrate/
#   ├── ro-crate-metadata.json      (metadata)
#   ├── ro-crate-preview.html       (human-readable preview)
#   └── data/
#       ├── engineering/
#       │   └── setup.dexpi.xml     (copied DEXPI)
#       └── experiments/
#           ├── exp_1/
#           │   ├── raw/            (copied raw data files)
#           │   ├── processed/      (copied processed data)
#           │   └── engineering/    (copied CAD/drawings)
#           └── exp_2/
#               └── ...
#
# This makes the RO-Crate fully self-contained and portable!
# ============================================================================

from tkinter import Tk, filedialog

# Initialize crate data structure with output_directory field
crate_data = {
    'output_directory': None,   # ← Will be set below (REQUIRED!)
    'general': {},              # General metadata
    'authors': [],              # List of contributors
    'dexpi': None,              # Single shared DEXPI file
    'experiments': []           # 0 to many experiments
}

# Current experiment being configured (temporary workspace)
current_experiment = {
    'id': '',
    'experimental_parameters': None,
    'analytical_files': [],
    'engineering_assets': []
}

# Widget to input or browse for output directory path
w_crate_dir = widgets.Text(
    value='./my_rocrate',
    placeholder='./my_rocrate',
    description='Output Dir:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px')
)

btn_browse_dir = widgets.Button(
    description='Browse...',
    button_style='',
    icon='folder-open',
    layout=widgets.Layout(width='120px')
)

btn_create_dir = widgets.Button(
    description='Create Directory',
    button_style='info',
    icon='folder',
    layout=widgets.Layout(width='150px')
)

out_crate_dir = widgets.Output()


def browse_directory(b):
    """Open file browser to select output directory."""
    root = Tk()
    root.withdraw()  # Hide the main window
    root.attributes('-topmost', True)  # Bring dialog to front
    
    folder_selected = filedialog.askdirectory(
        title='Select RO-Crate Output Directory',
        initialdir=os.path.expanduser('~')
    )
    
    root.destroy()
    
    if folder_selected:
        w_crate_dir.value = folder_selected
        with out_crate_dir:
            clear_output()
            print(f"📁 Selected: {folder_selected}")


def create_crate_directory(b):
    """Create the RO-Crate output directory structure."""
    with out_crate_dir:
        clear_output()
        crate_path = Path(w_crate_dir.value)
        
        try:
            # Create main directory and subdirectories
            crate_path.mkdir(parents=True, exist_ok=True)
            (crate_path / 'data' / 'engineering').mkdir(parents=True, exist_ok=True)
            (crate_path / 'data' / 'experiments').mkdir(parents=True, exist_ok=True)
            
            # Store absolute path in crate_data
            crate_data['output_directory'] = str(crate_path.absolute())
            
            print(f"✅ RO-Crate directory created: {crate_path.absolute()}")
            print(f"\n📁 Directory structure:")
            print(f"  {crate_path.name}/")
            print(f"  ├── data/")
            print(f"  │   ├── engineering/")
            print(f"  │   └── experiments/")
            print(f"  └── ro-crate-metadata.json (will be created on export)")
            print(f"\n💡 All referenced files will be COPIED here automatically!")
            print(f"   → Files can be located ANYWHERE on your system")
            print(f"   → Everything gets copied into this self-contained structure")
            
        except Exception as e:
            print(f"✗ Error creating directory: {str(e)}")


btn_browse_dir.on_click(browse_directory)
btn_create_dir.on_click(create_crate_directory)

# Display UI
display(HTML("<h3>📂 Select RO-Crate Output Directory</h3>"))
display(HTML("<p><strong style='color: #d73a49;'>⚠️ Start here!</strong> All files will be <strong>copied</strong> into this directory to create a self-contained, portable RO-Crate package.</p>"))
display(HTML("<p>💡 <em>Your source files can be located anywhere - they will be copied automatically!</em></p>"))
display(widgets.HBox([w_crate_dir, btn_browse_dir]))
display(btn_create_dir, out_crate_dir)

## 1. General RO-Crate Metadata

Enter basic information about your research object.

In [ ]:
# ============================================================================
# General Metadata Input Widgets
# ============================================================================
# Note: The crate_data structure is now initialized in Section 0 above,
# which includes the critical 'output_directory' field.
# ============================================================================

w_name = widgets.Text(
    value='',
    placeholder='e.g., Temperature Optimization Study',
    description='Crate Name:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px')
)

w_description = widgets.Textarea(
    value='',
    placeholder='Detailed description of the study and experiments...',
    description='Description:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px', height='80px')
)

w_keywords = widgets.Text(
    value='',
    placeholder='chemical engineering, FAIR, experimental data',
    description='Keywords:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px')
)

w_license = widgets.Dropdown(
    options=['CC-BY-4.0', 'CC-BY-SA-4.0', 'CC0-1.0', 'MIT', 'Apache-2.0'],
    value='CC-BY-4.0',
    description='License:',
    style={'description_width': '150px'}
)

w_date_created = widgets.Text(
    value=datetime.now().strftime('%Y-%m-%d'),
    description='Date Created:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

btn_save_general = widgets.Button(
    description='Save General Info',
    button_style='success',
    icon='check'
)

out_general = widgets.Output()


def save_general(b):
    """Save general metadata to crate_data structure."""
    with out_general:
        clear_output()
        # Store metadata in crate_data dictionary
        crate_data['general'] = {
            'name': w_name.value,
            'description': w_description.value,
            'keywords': [k.strip() for k in w_keywords.value.split(',') if k.strip()],
            'license': w_license.value,
            'dateCreated': w_date_created.value
        }
        print(f"✓ General metadata saved")
        print(f"  Crate: {crate_data['general']['name']}")


btn_save_general.on_click(save_general)

# Display UI components
display(HTML("<h3>General Information</h3>"))
display(w_name, w_description, w_keywords, w_license, w_date_created, btn_save_general, out_general)

Text(value='', description='Crate Name:', layout=Layout(width='600px'), placeholder='e.g., Temperature Optimiz…

Textarea(value='', description='Description:', layout=Layout(height='80px', width='600px'), placeholder='Detai…

Text(value='', description='Keywords:', layout=Layout(width='600px'), placeholder='chemical engineering, FAIR,…

Dropdown(description='License:', options=('CC-BY-4.0', 'CC-BY-SA-4.0', 'CC0-1.0', 'MIT', 'Apache-2.0'), style=…

Text(value='2026-03-23', description='Date Created:', layout=Layout(width='400px'), style=TextStyle(descriptio…

Button(button_style='success', description='Save General Info', icon='check', style=ButtonStyle())

Output()

### Add Authors and Contributors

In [22]:
# Author input widgets
w_author_name = widgets.Text(
    placeholder='Jane Doe',
    description='Name:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

w_author_orcid = widgets.Text(
    placeholder='0000-0000-0000-0000',
    description='ORCID:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

w_author_affiliation = widgets.Text(
    placeholder='University of Example',
    description='Affiliation:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

w_author_role = widgets.Dropdown(
    options=['Author', 'Contributor', 'DataCollector', 'DataCurator', 'ProjectLeader'],
    value='Author',
    description='Role:',
    style={'description_width': '120px'}
)

btn_add_author = widgets.Button(
    description='Add Author',
    button_style='primary',
    icon='plus'
)

out_authors = widgets.Output()

def add_author(b):
    with out_authors:
        if not w_author_name.value:
            print("⚠ Name is required")
            return
        
        author = {
            'name': w_author_name.value,
            'orcid': w_author_orcid.value if w_author_orcid.value else None,
            'affiliation': w_author_affiliation.value if w_author_affiliation.value else None,
            'role': w_author_role.value
        }
        crate_data['authors'].append(author)
        
        clear_output()
        print(f"✓ Added: {author['name']} ({author['role']})")
        print(f"  Total authors: {len(crate_data['authors'])}")
        
        # Clear input fields
        w_author_name.value = ''
        w_author_orcid.value = ''
        w_author_affiliation.value = ''

btn_add_author.on_click(add_author)

display(HTML("<h3>Authors and Contributors</h3>"))
display(w_author_name, w_author_orcid, w_author_affiliation, w_author_role, btn_add_author, out_authors)

Text(value='', description='Name:', layout=Layout(width='400px'), placeholder='Jane Doe', style=TextStyle(desc…

Text(value='', description='ORCID:', layout=Layout(width='400px'), placeholder='0000-0000-0000-0000', style=Te…

Text(value='', description='Affiliation:', layout=Layout(width='400px'), placeholder='University of Example', …

Dropdown(description='Role:', options=('Author', 'Contributor', 'DataCollector', 'DataCurator', 'ProjectLeader…

Button(button_style='primary', description='Add Author', icon='plus', style=ButtonStyle())

Output()

## 2. Shared Experimental Setup (DEXPI)

Load your DEXPI P&ID file that describes the experimental setup. This will be shared across all experiments.

In [ ]:
# ============================================================================
# Load Shared DEXPI File
# ============================================================================
# This section loads the DEXPI P&ID XML file that describes the experimental
# setup. This single DEXPI file is shared across ALL experiments - different
# experiments may have different operating parameters or data files, but use
# the same physical setup described in this DEXPI.
#
# The pyDEXPI ProteusSerializer loads and parses the XML into a Python object
# model, which is then processed by our helper functions to extract equipment
# and instrument metadata.
# ============================================================================

w_dexpi_path = widgets.Text(
    value='examples/C01V04-VER.EX01.xml',
    placeholder='examples/C01V04-VER.EX01.xml',
    description='DEXPI Path:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

btn_browse_dexpi = widgets.Button(
    description='Browse...',
    button_style='',
    icon='file',
    layout=widgets.Layout(width='100px')
)

btn_load_dexpi = widgets.Button(
    description='Load DEXPI',
    button_style='info',
    icon='upload'
)

out_dexpi = widgets.Output()


def browse_dexpi_file(b):
    """Open file browser to select DEXPI file."""
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    
    file_selected = filedialog.askopenfilename(
        title='Select DEXPI XML File',
        filetypes=[('XML files', '*.xml'), ('All files', '*.*')],
        initialdir=os.path.expanduser('~')
    )
    
    root.destroy()
    
    if file_selected:
        w_dexpi_path.value = file_selected
        with out_dexpi:
            clear_output()
            print(f"📄 Selected: {file_selected}")


def load_dexpi(b):
    """Load DEXPI XML file using pyDEXPI and extract metadata."""
    with out_dexpi:
        clear_output()
        try:
            # Validate file exists
            if not Path(w_dexpi_path.value).exists():
                print(f"⚠ File not found: {w_dexpi_path.value}")
                return
            
            # Load DEXPI using pyDEXPI ProteusSerializer
            # ProteusSerializer handles Proteus-compatible DEXPI XML format
            loader = ProteusSerializer()
            directory_path = str(Path(w_dexpi_path.value).parent)
            filename = Path(w_dexpi_path.value).name
            
            # Parse DEXPI XML into Python object model
            dexpi_model = loader.load(directory_path, filename)
            
            # Extract equipment and instruments using helper functions
            equipment = extract_equipment(dexpi_model)
            instruments = extract_instruments(dexpi_model)
            
            # Store in crate_data (shared across all experiments)
            crate_data['dexpi'] = {
btn_browse_dexpi.on_click(browse_dexpi_file)
btn_load_dexpi.on_click(load_dexpi)

# Display UI components
display(HTML("<h3>Shared DEXPI P&ID File</h3>"))
display(HTML("<p>This DEXPI file defines the experimental setup and will be shared across all experiments.</p>"))

display(HTML("<p>💡 <em>The file can be located anywhere - it will be copied into the RO-Crate.</em></p>"))
            print(f"✓ DEXPI loaded: {Path(w_dexpi_path.value).name}")display(w_dexpi_path, btn_load_dexpi, out_dexpi)

display(widgets.HBox([w_dexpi_path, btn_browse_dexpi]))
            print(f"  Equipment items: {len(equipment)}")display(HTML("<p>This DEXPI file defines the experimental setup and will be shared across all experiments.</p>"))

display(btn_load_dexpi, out_dexpi)
            print(f"  Instruments: {len(instruments)}")display(HTML("<h3>Shared DEXPI P&ID File</h3>"))

            # Display UI components

            # Show sample equipment

            if equipment:btn_load_dexpi.on_click(load_dexpi)

                print("\n  Sample equipment:")

                for eq in list(equipment.values())[:5]:

                    print(f"    - {eq['tag_name']}: {eq['type']}")            traceback.print_exc()

                        import traceback

            # Show sample instruments            print(f"✗ Error loading DEXPI: {str(e)}")

            if instruments:        except Exception as e:

                print("\n  Sample instruments:")                    

                for inst in list(instruments.values())[:5]:                    print(f"    - {inst['tag_name']}: {inst.get('description', 'N/A')}")

Text(value='examples/C01V04-VER.EX01.xml', description='DEXPI Path:', layout=Layout(width='600px'), placeholde…

Button(button_style='info', description='Load DEXPI', icon='upload', style=ButtonStyle())

Output()

## 3. Configure Experiments (0 to many)

Configure individual experiments. Each experiment can have different parameters and data files, but shares the same DEXPI setup.

In [ ]:
### Experiment Configuration ###

import yaml

# Experiment ID and name
w_exp_id = widgets.Text(
    value='exp_1',
    placeholder='exp_1',
    description='Experiment ID:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

w_exp_name = widgets.Text(
    value='',
    placeholder='Baseline Experiment',
    description='Experiment Name:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px')
)

# Parameters file
w_params_path = widgets.Text(
    value='',
    placeholder='templates/experiment_parameters.yaml',
    description='Parameters:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px')
)

btn_load_params = widgets.Button(
    description='Load Parameters',
    button_style='info',
    icon='upload'
)

out_exp_config = widgets.Output()

def load_params(b):
    with out_exp_config:
        clear_output()
        try:
            if not Path(w_params_path.value).exists():
                print(f"⚠ File not found: {w_params_path.value}")
                return
            
            with open(w_params_path.value, 'r') as f:
                params = yaml.safe_load(f)
            
            current_experiment['experimental_parameters'] = params
            
            print(f"✓ Parameters loaded for {w_exp_id.value}")
            print(f"  Experiment: {params.get('experiment', {}).get('title', 'N/A')}")
            
            if params.get('conditions'):
                print(f"  Conditions: {len(params['conditions'])} parameters defined")
                
        except Exception as e:
            print(f"✗ Error loading parameters: {str(e)}")

btn_load_params.on_click(load_params)

display(HTML("<h3>Current Experiment Configuration</h3>"))
display(widgets.HBox([w_exp_id, w_exp_name]))
display(w_params_path, btn_load_params, out_exp_config)

### Add Data Files for Current Experiment

In [24]:
### Add Analytical Data Files ###

def get_instrument_options():
    dexpi = crate_data.get('dexpi')
    if not dexpi or not dexpi.get('instruments'):
        return [('Load DEXPI first', None)]
    
    options = []
    for inst_id, inst_data in dexpi['instruments'].items():
        label = f"{inst_data['tag_name']} - {inst_data.get('description', 'N/A')}"
        options.append((label, inst_id))
    return options

w_data_path = widgets.Text(
    value='',
    placeholder='data/raw/gc_sample_001.csv',
    description='Data File:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px')
)

w_instrument_select = widgets.Dropdown(
    options=get_instrument_options(),
    description='Instrument:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px')
)

w_data_type = widgets.Dropdown(
    options=['RawData', 'ProcessedData', 'CalibrationData', 'DerivedData'],
    value='RawData',
    description='Data Type:',
    style={'description_width': '150px'}
)

w_data_description = widgets.Textarea(
    placeholder='Description of the data file...',
    description='Description:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px', height='80px')
)

btn_add_data = widgets.Button(
    description='Add Data File',
    button_style='success',
    icon='plus'
)

btn_refresh_instruments = widgets.Button(
    description='Refresh Instruments',
    button_style='warning',
    icon='refresh'
)

out_data = widgets.Output()

def refresh_instruments(b):
    w_instrument_select.options = get_instrument_options()
    with out_data:
        clear_output(wait=True)
        print("✓ Instrument list refreshed from shared DEXPI")

def add_data_file(b):
    with out_data:
        clear_output()
        if not w_data_path.value:
            print("⚠ Data file path is required")
            return
        
        if not w_instrument_select.value:
            print("⚠ Please select an instrument (load DEXPI first)")
            return
        
        file_entry = {
            'path': w_data_path.value,
            'instrument_id': w_instrument_select.value,
            'data_type': w_data_type.value,
            'description': w_data_description.value
        }
        
        current_experiment['analytical_files'].append(file_entry)
        
        print(f"✓ Added data file: {file_entry['path']}")
        print(f"  Instrument: {w_instrument_select.label}")
        print(f"  Type: {file_entry['data_type']}")
        print(f"  Total files for {w_exp_id.value}: {len(current_experiment['analytical_files'])}")
        
        # Clear inputs
        w_data_path.value = ''
        w_data_description.value = ''

btn_add_data.on_click(add_data_file)
btn_refresh_instruments.on_click(refresh_instruments)

display(HTML("<h4>Analytical Data Files</h4>"))
display(w_data_path, w_instrument_select, w_data_type, w_data_description,
        widgets.HBox([btn_add_data, btn_refresh_instruments]), out_data)

Text(value='', description='Data File:', layout=Layout(width='600px'), placeholder='data/raw/gc_sample_001.csv…

Dropdown(description='Instrument:', layout=Layout(width='600px'), options=(('PV4712.02 - ', '604be6a4-93cc-400…

Dropdown(description='Data Type:', options=('RawData', 'ProcessedData', 'CalibrationData', 'DerivedData'), sty…

Textarea(value='', description='Description:', layout=Layout(height='80px', width='600px'), placeholder='Descr…

Output()

### Add Engineering Assets for Current Experiment

In [ ]:
### Add Engineering Assets ###

def get_equipment_options():
    dexpi = crate_data.get('dexpi')
    if not dexpi or not dexpi.get('equipment'):
        return [('Load DEXPI first', None)]
    
    options = []
    for eq_id, eq_data in dexpi['equipment'].items():
        label = f"{eq_data['tag_name']} - {eq_data.get('type', 'N/A')}"
        options.append((label, eq_id))
    return options

w_asset_path = widgets.Text(
    value='',
    placeholder='data/engineering/reactor_photo.jpg',
    description='Asset Path:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px')
)

w_equipment_select = widgets.Dropdown(
    options=get_equipment_options(),
    description='Equipment:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px')
)

w_asset_type = widgets.Dropdown(
    options=['CADModel', 'TechnicalDrawing', 'GeometryFile', 'Specification', 'Photo'],
    value='Photo',
    description='Asset Type:',
    style={'description_width': '150px'}
)

w_asset_description = widgets.Textarea(
    placeholder='Description of the engineering asset...',
    description='Description:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px', height='80px')
)

btn_add_asset = widgets.Button(
    description='Add Engineering Asset',
    button_style='success',
    icon='plus'
)

btn_refresh_equipment = widgets.Button(
    description='Refresh Equipment',
    button_style='warning',
    icon='refresh'
)

out_assets = widgets.Output()

def refresh_equipment(b):
    w_equipment_select.options = get_equipment_options()
    with out_assets:
        clear_output(wait=True)
        print("✓ Equipment list refreshed from shared DEXPI")

def add_engineering_asset(b):
    with out_assets:
        clear_output()
        if not w_asset_path.value:
            print("⚠ Asset path is required")
            return
        
        if not w_equipment_select.value:
            print("⚠ Please select equipment (load DEXPI first)")
            return
        
        asset_entry = {
            'path': w_asset_path.value,
            'equipment_id': w_equipment_select.value,
            'asset_type': w_asset_type.value,
            'description': w_asset_description.value
        }
        
        current_experiment['engineering_assets'].append(asset_entry)
        
        print(f"✓ Added engineering asset: {asset_entry['path']}")
        print(f"  Linked to: {w_equipment_select.label}")
        print(f"  Type: {asset_entry['asset_type']}")
        print(f"  Total assets for {w_exp_id.value}: {len(current_experiment['engineering_assets'])}")
        
        # Clear inputs
        w_asset_path.value = ''
        w_asset_description.value = ''

btn_add_asset.on_click(add_engineering_asset)
btn_refresh_equipment.on_click(refresh_equipment)

display(HTML("<h4>Engineering Assets</h4>"))
display(w_asset_path, w_equipment_select, w_asset_type, w_asset_description,
        widgets.HBox([btn_add_asset, btn_refresh_equipment]), out_assets)

In [ ]:
### Add Current Experiment to RO-Crate ###

btn_add_experiment = widgets.Button(
    description='Add Experiment to Crate',
    button_style='primary',
    icon='check',
    layout=widgets.Layout(width='250px')
)

btn_reset_experiment = widgets.Button(
    description='Reset for New Experiment',
    button_style='warning',
    icon='refresh'
)

btn_skip_experiments = widgets.Button(
    description='Skip (No Experiments)',
    button_style='',
    icon='forward'
)

out_add_exp = widgets.Output()

def add_experiment_to_crate(b):
    with out_add_exp:
        clear_output()
        
        if not w_exp_id.value:
            print("⚠ Experiment ID is required")
            return
        
        # Create experiment entry
        exp_entry = {
            'id': w_exp_id.value,
            'experimental_parameters': current_experiment.get('experimental_parameters'),
            'analytical_files': current_experiment['analytical_files'].copy(),
            'engineering_assets': current_experiment['engineering_assets'].copy()
        }
        
        crate_data['experiments'].append(exp_entry)
        
        print(f"✓ Experiment '{w_exp_id.value}' added to crate")
        if exp_entry['experimental_parameters']:
            print(f"  Parameters: {exp_entry['experimental_parameters'].get('experiment', {}).get('title', 'N/A')}")
        print(f"  Analytical files: {len(exp_entry['analytical_files'])}")
        print(f"  Engineering assets: {len(exp_entry['engineering_assets'])}")
        print(f"\n  Total experiments in crate: {len(crate_data['experiments'])}")
        print("\n  ⚠ Remember to reset for next experiment!")

def reset_current_experiment(b):
    global current_experiment
    current_experiment = {
        'id': '',
        'experimental_parameters': None,
        'analytical_files': [],
        'engineering_assets': []
    }
    
    # Update ID to next available
    next_num = len(crate_data['experiments']) + 1
    w_exp_id.value = f'exp_{next_num}'
    w_params_path.value = ''
    
    with out_add_exp:
        clear_output()
        print(f"✓ Reset complete. Ready for experiment {next_num}")

def skip_experiments(b):
    with out_add_exp:
        clear_output()
        print("ℹ️ Skipping experiments - RO-Crate will contain only setup (DEXPI)")
        print("   This is useful for documenting experimental setups before running experiments")

btn_add_experiment.on_click(add_experiment_to_crate)
btn_reset_experiment.on_click(reset_current_experiment)
btn_skip_experiments.on_click(skip_experiments)

display(HTML("<h4>Finalize Experiment</h4>"))
display(HTML("<p><strong>Tip:</strong> You can add 0 to many experiments. Click 'Add Experiment to Crate', then 'Reset for New Experiment' to add more.</p>"))
display(widgets.HBox([btn_add_experiment, btn_reset_experiment, btn_skip_experiments]), out_add_exp)

### Finalize Current Experiment and Start New One

## 4. Review and Export RO-Crate

Review your metadata and export the RO-Crate.

In [ ]:
### Review Metadata ###

btn_review = widgets.Button(
    description='Review Metadata',
    button_style='info',
    icon='eye'
)

out_review = widgets.Output()

def review_metadata(b):
    with out_review:
        clear_output()
        print("=" * 70)
        print("RO-CRATE METADATA REVIEW")
        print("=" * 70)
        
        print(f"\n📦 GENERAL INFORMATION")
        print(f"  Name: {crate_data['general'].get('name', 'Not set')}")
        print(f"  Keywords: {', '.join(crate_data['general'].get('keywords', []))}")
        print(f"  License: {crate_data['general'].get('license', 'Not set')}")
        
        print(f"\n👥 AUTHORS: {len(crate_data['authors'])}")
        for author in crate_data['authors']:
            print(f"  - {author['name']} ({author['role']})")
        
        print(f"\n🏭 SHARED EXPERIMENTAL SETUP")
        if crate_data.get('dexpi'):
            print(f"  DEXPI: {Path(crate_data['dexpi']['path']).name}")
            print(f"  Equipment: {len(crate_data['dexpi'].get('equipment', {}))} items")
            print(f"  Instruments: {len(crate_data['dexpi'].get('instruments', {}))} items")
        else:
            print("  ⚠ No DEXPI loaded")
        
        print(f"\n🔬 EXPERIMENTS: {len(crate_data['experiments'])}")
        if not crate_data['experiments']:
            print("  ℹ️ No experiments defined (setup-only RO-Crate)")
        else:
            for i, exp in enumerate(crate_data['experiments'], 1):
                print(f"\n  Experiment {i}: {exp['id']}")
                if exp.get('experimental_parameters'):
                    title = exp['experimental_parameters'].get('experiment', {}).get('title', 'N/A')
                    print(f"    Title: {title}")
                print(f"    Analytical files: {len(exp['analytical_files'])}")
                print(f"    Engineering assets: {len(exp['engineering_assets'])}")
        
        print("\n" + "=" * 70)

btn_review.on_click(review_metadata)

display(HTML("<h3>Review Metadata</h3>"))
display(btn_review, out_review)

### Export RO-Crate

In [ ]:
w_output_dir = widgets.Text(
    value='.',
    placeholder='.',
    description='Output Dir:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

w_validate = widgets.Checkbox(
    value=True,
    description='Validate after export',
    style={'description_width': 'initial'}
)

btn_export = widgets.Button(
    description='Export RO-Crate',
    button_style='success',
    icon='download',
    layout=widgets.Layout(width='200px')
)

out_export = widgets.Output()

def export_crate(b):
    with out_export:
        clear_output()
        try:
            print("🚀 Building RO-Crate...")
            
            # Build the crate using helper class
            builder = OPSEECrateBuilder()
            crate = builder.build_crate(crate_data)
            
            # Write to disk
            output_path = w_output_dir.value
            print(f"📝 Writing to {output_path}/ro-crate-metadata.json...")
            crate.write(output_path)
            
            print("\n✅ RO-Crate exported successfully!")
            print(f"\n📄 Files created:")
            print(f"  - {output_path}/ro-crate-metadata.json")
            print(f"  - {output_path}/ro-crate-preview.html")
            
            # Validate if requested
            if w_validate.value:
                print("\n🔍 Validating RO-Crate...")
                validation_result = validate_crate(f"{output_path}/ro-crate-metadata.json")
                if validation_result['valid']:
                    print("✓ Validation passed")
                else:
                    print("⚠ Validation warnings:")
                    for warning in validation_result.get('warnings', []):
                        print(f"  - {warning}")
            
            print("\n" + "="*60)
            print("Next steps:")
            print("  1. Review ro-crate-metadata.json")
            print("  2. Open ro-crate-preview.html in a browser")
            print("  3. Share or publish your RO-Crate")
            print("="*60)
            
        except Exception as e:
            print(f"\n✗ Error exporting crate: {str(e)}")
            import traceback
            traceback.print_exc()

btn_export.on_click(export_crate)

display(HTML("<h3>Export RO-Crate</h3>"))
display(w_output_dir, w_validate, btn_export, out_export)

---

## Summary

You have successfully created an RO-Crate for your chemical engineering experiments! 

### Key Features

- **Unified Setup**: Single shared DEXPI file describes experimental setup
- **Flexible Experiments**: Support for 0 to many experiments on the same setup
- **FAIR Principles**: Complete metadata following FAIR principles
- **pydexpi Integration**: Robust DEXPI XML parsing with pydexpi library
- **Semantic Links**: Data files linked to instruments and equipment
- **Provenance**: Complete experimental provenance information

### RO-Crate Structure

```
ro-crate/
├── ro-crate-metadata.json
└── data/
    ├── engineering/
    │   └── setup.dexpi.xml          # Shared DEXPI file
    └── experiments/
        ├── exp_1/                    # Experiment 1 data
        │   ├── raw/
        │   ├── processed/
        │   └── engineering/
        └── exp_2/                    # Experiment 2 data
            ├── raw/
            ├── processed/
            └── engineering/
```

### Resources

- [RO-Crate Specification](https://www.researchobject.org/ro-crate/)
- [pydexpi Documentation](https://github.com/muellergroup/pydexpi)
- [DEXPI Standard](https://www.dexpi.org/)
- [OPSEE Documentation](docs/WORKFLOW.md)